In [1]:
import pandas as pd
import numpy as np

master_df = pd.read_parquet(
    "../data/processed/master_df.parquet"
)

master_df.shape

(9959019, 45)

In [2]:
forecast_df = master_df[

    [

        "date",

        "sku_id",

        "quantity",

        "promo_id",

        "category",

        "subcategory",

        "brand",

        "month",

        "quarter",

        "weekday",

        "week"

    ]

].copy()

forecast_df.head()

,date,sku_id,quantity,promo_id,category,subcategory,brand,month,quarter,weekday,week
0,2025-04-02,SKU02498,1,None,Apparel & Footwear,Men's Wear,GreenValley,4,2,Wednesday,14
1,2025-04-02,SKU04596,3,None,Dairy & Bakery,Eggs,ValueChoice,4,2,Wednesday,14
2,2022-04-24,SKU00078,1,None,Frozen Foods,Ready Meals,NatureBlend,4,2,Sunday,16
3,2022-04-24,SKU00554,2,None,Snacks & Confectionery,Chocolates,TrueTaste,4,2,Sunday,16
4,2024-09-22,SKU03727,1,None,Electronics & Accessories,Mobile Accessories,QuickBite,9,3,Sunday,38


In [5]:
weekly_sales = (

    forecast_df

    .groupby(

        [

            "sku_id",

            pd.Grouper(

                key="date",

                freq="W"

            )

        ]

    )

    .agg(

        {

            "quantity": "sum",

            "promo_id": "count"

        }

    )

    .reset_index()

)

weekly_sales.rename(

    columns={

        "promo_id": "promo_count"

    },

    inplace=True

)

weekly_sales.head()

,sku_id,date,quantity,promo_count
0,SKU00001,2022-01-02,3,0
1,SKU00001,2022-01-09,14,0
2,SKU00001,2022-01-16,29,0
3,SKU00001,2022-01-23,12,0
4,SKU00001,2022-01-30,15,0


In [6]:
weekly_sales["year"] = weekly_sales["date"].dt.year

weekly_sales["month"] = weekly_sales["date"].dt.month

weekly_sales["quarter"] = weekly_sales["date"].dt.quarter

weekly_sales["weekofyear"] = weekly_sales["date"].dt.isocalendar().week

In [9]:
weekly_sales["lag_1"] = (

    weekly_sales

    .groupby("sku_id")["quantity"]

    .shift(1)

)

In [11]:
weekly_sales["lag_2"] = (

    weekly_sales

    .groupby("sku_id")["quantity"]

    .shift(2)

)

In [13]:
weekly_sales["lag_4"] = (

    weekly_sales

    .groupby("sku_id")["quantity"]

    .shift(4)

)

In [15]:
weekly_sales["rolling_mean_4"] = (

    weekly_sales

    .groupby("sku_id")["quantity"]

    .transform(

        lambda x: x.rolling(4).mean()

    )

)

In [17]:
weekly_sales["rolling_mean_8"] = (

    weekly_sales

    .groupby("sku_id")["quantity"]

    .transform(

        lambda x: x.rolling(8).mean()

    )

)

In [19]:
weekly_sales = weekly_sales.dropna()

weekly_sales.shape

(998143, 13)

In [21]:
weekly_sales.to_parquet(

    "../data/processed/weekly_sales.parquet",

    index=False

)

In [23]:
weekly_sales.head()

,sku_id,date,quantity,promo_count,year,month,quarter,weekofyear,lag_1,lag_2,lag_4,rolling_mean_4,rolling_mean_8
7,SKU00001,2022-02-20,29,0,2022,2,1,7,28.0,13.0,12.0,21.25,17.875
8,SKU00001,2022-02-27,5,0,2022,2,1,8,29.0,28.0,15.0,18.75,18.125
9,SKU00001,2022-03-06,32,0,2022,3,1,9,5.0,29.0,13.0,23.50,20.375
10,SKU00001,2022-03-13,12,0,2022,3,1,10,32.0,5.0,28.0,19.50,18.250
11,SKU00001,2022-03-20,11,0,2022,3,1,11,12.0,32.0,29.0,15.00,18.125


In [25]:
weekly_sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 998143 entries, 7 to 1033142
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   sku_id          998143 non-null  object        
 1   date            998143 non-null  datetime64[ns]
 2   quantity        998143 non-null  int64         
 3   promo_count     998143 non-null  int64         
 4   year            998143 non-null  int32         
 5   month           998143 non-null  int32         
 6   quarter         998143 non-null  int32         
 7   weekofyear      998143 non-null  UInt32        
 8   lag_1           998143 non-null  float64       
 9   lag_2           998143 non-null  float64       
 10  lag_4           998143 non-null  float64       
 11  rolling_mean_4  998143 non-null  float64       
 12  rolling_mean_8  998143 non-null  float64       
dtypes: UInt32(1), datetime64[ns](1), float64(5), int32(3), int64(2), object(1)
memory usage: 92.3

In [27]:
weekly_sales["has_promo"] = (

    weekly_sales["promo_count"] > 0

).astype(int)

In [29]:
weekly_sales["lag_8"] = (

    weekly_sales

    .groupby("sku_id")["quantity"]

    .shift(8)

)

In [31]:
weekly_sales["rolling_std_4"] = (

    weekly_sales

    .groupby("sku_id")["quantity"]

    .transform(

        lambda x: x.rolling(4).std()

    )

)

In [33]:
weekly_sales["expanding_mean"] = (

    weekly_sales

    .groupby("sku_id")["quantity"]

    .transform(

        lambda x: x.expanding().mean()

    )

)

In [35]:
features = [

    "promo_count",

    "has_promo",

    "year",

    "month",

    "quarter",

    "weekofyear",

    "lag_1",

    "lag_2",

    "lag_4",

    "lag_8",

    "rolling_mean_4",

    "rolling_mean_8",

    "rolling_std_4",

    "expanding_mean"

]

In [37]:
weekly_sales["seasonal_naive"] = (

    weekly_sales

    .groupby("sku_id")["quantity"]

    .shift(52)

)

In [39]:
weekly_sales = weekly_sales.dropna()

In [41]:
weekly_sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 738143 entries, 59 to 1033142
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   sku_id          738143 non-null  object        
 1   date            738143 non-null  datetime64[ns]
 2   quantity        738143 non-null  int64         
 3   promo_count     738143 non-null  int64         
 4   year            738143 non-null  int32         
 5   month           738143 non-null  int32         
 6   quarter         738143 non-null  int32         
 7   weekofyear      738143 non-null  UInt32        
 8   lag_1           738143 non-null  float64       
 9   lag_2           738143 non-null  float64       
 10  lag_4           738143 non-null  float64       
 11  rolling_mean_4  738143 non-null  float64       
 12  rolling_mean_8  738143 non-null  float64       
 13  has_promo       738143 non-null  int64         
 14  lag_8           738143 non-null  float6

In [43]:
weekly_sales.to_parquet(
    "../data/processed/weekly_sales.parquet",
    index=False
)